In [12]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
from selenium.webdriver.chrome.options import Options
import time
import pandas as pd

# ---- 설정 ----
USER_ID = "leewu08"        # ← 여기에 실제 아이디
USER_PW = "Emporio119"  # ← 여기에 실제 비밀번호

options = Options()
options.add_argument("--start-maximized")
# options.add_argument("--headless")  # 필요 시 활성화

driver = webdriver.Chrome(options=options)

try:
    # 1. 로그인 페이지 진입
    driver.get("https://www.kpi.or.kr/www/price/category.asp?CATE_CD=104501")
    time.sleep(2)

    # 2. 로그인 버튼 클릭
    login_button = driver.find_element(By.XPATH, '//a[contains(text(), "로그인")]')
    login_button.click()
    time.sleep(2)

    # 3. 로그인 정보 입력
    driver.find_element(By.CSS_SELECTOR, "#user_id").send_keys(USER_ID)
    driver.find_element(By.CSS_SELECTOR, "#user_pw").send_keys(USER_PW)
    driver.find_element(By.CSS_SELECTOR, "#sendLogin").click()
    time.sleep(3)

    # 4. 로그인 완료 후 절연전선 페이지 재진입
    driver.get("https://www.kpi.or.kr/www/price/category.asp?CATE_CD=104501")
    time.sleep(2)

    # 5. 제품 정보 크롤링
    items = driver.find_elements(By.CSS_SELECTOR, "div.set-mitem")
    results = []

    for item in items:
        try:
            # openGoods('코드') 추출
            a_tag = item.find_element(By.TAG_NAME, "a")
            href = a_tag.get_attribute("href")
            code = href.split("openGoods('")[1].split("')")[0]

            # 제품명
            title = a_tag.get_attribute("title")

            # 이미지 URL
            img_tag = item.find_element(By.CSS_SELECTOR, "li.set-img img")
            img_url = img_tag.get_attribute("src")
            if img_url.startswith("/"):
                img_url = "https://www.kpi.or.kr" + img_url

            results.append({
                "제품코드": code,
                "제품명": title,
                "이미지": img_url
            })
        except Exception as e:
            print(f"[!] 항목 처리 중 오류: {e}")
            continue

    # 6. 저장
    df = pd.DataFrame(results)
    df.to_csv("kpi_절연전선_상품목록.csv", index=False, encoding="utf-8-sig")
    print("✅ 저장 완료: kpi_절연전선_상품목록.csv")

finally:
    driver.quit()

✅ 저장 완료: kpi_절연전선_상품목록.csv


In [14]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time
import pandas as pd

# --- 설정 ---
USER_ID = "leewu08"
USER_PW = "Emporio119"

options = Options()
options.add_argument("--start-maximized")
# options.add_argument("--headless")  # 필요시

driver = webdriver.Chrome(options=options)

try:
    # 1. 로그인 페이지 진입
    driver.get("https://www.kpi.or.kr/www/price/category.asp?CATE_CD=104501")
    time.sleep(2)

    # 2. 로그인
    driver.find_element(By.XPATH, '//a[contains(text(), "로그인")]').click()
    time.sleep(2)
    driver.find_element(By.CSS_SELECTOR, "#user_id").send_keys(USER_ID)
    driver.find_element(By.CSS_SELECTOR, "#user_pw").send_keys(USER_PW)
    driver.find_element(By.CSS_SELECTOR, "#sendLogin").click()
    time.sleep(3)

    # 3. 카테고리 목록 가져오기
    driver.get("https://www.kpi.or.kr/www/price/category.asp?CATE_CD=104501")
    time.sleep(2)
    
    category_links = driver.find_elements(By.CSS_SELECTOR, "#location > div.dep-two > dl > dd > ul li a")
    category_list = []
    for link in category_links:
        name = link.text.strip()
        href = link.get_attribute("href")
        if "CATE_CD=" in href:
            cate_cd = href.split("CATE_CD=")[-1]
            category_list.append({"name": name, "cate_cd": cate_cd})

    print(f"📦 카테고리 {len(category_list)}개 수집됨")

    results = []

    # 4. 카테고리별 순회
    for cat in category_list:
        print(f"\n🔍 크롤링 중: {cat['name']} (CATE_CD={cat['cate_cd']})")
        driver.get(f"https://www.kpi.or.kr/www/price/category.asp?CATE_CD={cat['cate_cd']}")
        time.sleep(2)

        items = driver.find_elements(By.CSS_SELECTOR, "#table_two_sub > div.goods-item div.set-mitem")
        
        for item in items:
            try:
                a_tag = item.find_element(By.TAG_NAME, "a")
                href = a_tag.get_attribute("href")
                code = href.split("openGoods('")[1].split("')")[0]
                title = a_tag.get_attribute("title")

                img_tag = item.find_element(By.CSS_SELECTOR, "li.set-img img")
                img_url = img_tag.get_attribute("src")
                if img_url.startswith("/"):
                    img_url = "https://www.kpi.or.kr" + img_url

                results.append({
                    "카테고리": cat["name"],
                    "제품코드": code,
                    "제품명": title,
                    "이미지": img_url
                })
            except Exception as e:
                print(f"[!] {cat['name']} 상품 오류: {e}")
                continue

    # 5. 저장
    df = pd.DataFrame(results)
    df.to_csv("kpi_전기자재_카테고리별_상품.csv", index=False, encoding="utf-8-sig")
    print("\n✅ 저장 완료: kpi_전기자재_카테고리별_상품.csv")

finally:
    driver.quit()


📦 카테고리 26개 수집됨

🔍 크롤링 중:  (CATE_CD=104501)

🔍 크롤링 중:  (CATE_CD=104504)

🔍 크롤링 중:  (CATE_CD=104507)

🔍 크롤링 중:  (CATE_CD=104513)

🔍 크롤링 중:  (CATE_CD=104516)

🔍 크롤링 중:  (CATE_CD=104587)

🔍 크롤링 중:  (CATE_CD=104521)

🔍 크롤링 중:  (CATE_CD=104524)

🔍 크롤링 중:  (CATE_CD=104526)

🔍 크롤링 중:  (CATE_CD=104530)

🔍 크롤링 중:  (CATE_CD=104533)

🔍 크롤링 중:  (CATE_CD=104569)

🔍 크롤링 중:  (CATE_CD=104539)

🔍 크롤링 중:  (CATE_CD=104542)

🔍 크롤링 중:  (CATE_CD=104545)

🔍 크롤링 중:  (CATE_CD=104548)

🔍 크롤링 중:  (CATE_CD=104551)

🔍 크롤링 중:  (CATE_CD=104554)

🔍 크롤링 중:  (CATE_CD=104557)

🔍 크롤링 중:  (CATE_CD=104560)

🔍 크롤링 중:  (CATE_CD=104563)

🔍 크롤링 중:  (CATE_CD=104566)

🔍 크롤링 중:  (CATE_CD=104573)

🔍 크롤링 중:  (CATE_CD=104577)

🔍 크롤링 중:  (CATE_CD=104580)

🔍 크롤링 중:  (CATE_CD=104586)

✅ 저장 완료: kpi_전기자재_카테고리별_상품.csv


In [15]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time
import pandas as pd

# --- 설정 ---
USER_ID = "leewu08"       # ← 여기에 본인 ID
USER_PW = "Emporio119" # ← 여기에 본인 PW

options = Options()
options.add_argument("--start-maximized")
# options.add_argument("--headless")  # 필요 시 사용

driver = webdriver.Chrome(options=options)

try:
    # 1. 로그인 페이지 진입
    driver.get("https://www.kpi.or.kr/www/price/category.asp?CATE_CD=104501")
    time.sleep(2)

    # 2. 로그인
    driver.find_element(By.XPATH, '//a[contains(text(), "로그인")]').click()
    time.sleep(2)
    driver.find_element(By.CSS_SELECTOR, "#user_id").send_keys(USER_ID)
    driver.find_element(By.CSS_SELECTOR, "#user_pw").send_keys(USER_PW)
    driver.find_element(By.CSS_SELECTOR, "#sendLogin").click()
    time.sleep(3)

    # 3. 카테고리 목록 추출
    driver.get("https://www.kpi.or.kr/www/price/category.asp?CATE_CD=104501")
    time.sleep(2)
    
    category_links = driver.find_elements(By.CSS_SELECTOR, "#location > div.dep-two > dl > dd > ul li a")
    category_list = []
    for link in category_links:
        name = link.text.strip()
        href = link.get_attribute("href")
        if "CATE_CD=" in href:
            cate_cd = href.split("CATE_CD=")[-1]
            category_list.append({"name": name, "cate_cd": cate_cd})

    print(f"\n📦 총 {len(category_list)}개 카테고리 수집됨")

    results = []

    # 4. 카테고리별 순회하며 상품 정보 수집
    for cat in category_list:
        print(f"\n🔍 카테고리 크롤링 중: {cat['name']} (CATE_CD={cat['cate_cd']})")
        driver.get(f"https://www.kpi.or.kr/www/price/category.asp?CATE_CD={cat['cate_cd']}")
        time.sleep(2)

        items = driver.find_elements(By.CSS_SELECTOR, "#table_two_sub > div.goods-item div.set-mitem")
        
        for item in items:
            try:
                a_tag = item.find_element(By.TAG_NAME, "a")
                href = a_tag.get_attribute("href")
                code = href.split("openGoods('")[1].split("')")[0]
                title = a_tag.get_attribute("title")

                img_tag = item.find_element(By.CSS_SELECTOR, "li.set-img img")
                img_url = img_tag.get_attribute("src")
                if img_url.startswith("/"):
                    img_url = "https://www.kpi.or.kr" + img_url

                results.append({
                    "카테고리": cat["name"],
                    "제품코드": code,
                    "제품명": title,
                    "이미지": img_url
                })
            except Exception as e:
                print(f"[!] {cat['name']} 상품 오류: {e}")
                continue

    # 5. 저장
    df = pd.DataFrame(results)
    df.to_csv("kpi_전기자재_카테고리별_상품.csv", index=False, encoding="utf-8-sig")
    print("\n✅ 저장 완료: kpi_전기자재_카테고리별_상품.csv")

finally:
    driver.quit()



📦 총 26개 카테고리 수집됨

🔍 카테고리 크롤링 중:  (CATE_CD=104501)

🔍 카테고리 크롤링 중:  (CATE_CD=104504)

🔍 카테고리 크롤링 중:  (CATE_CD=104507)

🔍 카테고리 크롤링 중:  (CATE_CD=104513)

🔍 카테고리 크롤링 중:  (CATE_CD=104516)

🔍 카테고리 크롤링 중:  (CATE_CD=104587)

🔍 카테고리 크롤링 중:  (CATE_CD=104521)

🔍 카테고리 크롤링 중:  (CATE_CD=104524)

🔍 카테고리 크롤링 중:  (CATE_CD=104526)

🔍 카테고리 크롤링 중:  (CATE_CD=104530)

🔍 카테고리 크롤링 중:  (CATE_CD=104533)

🔍 카테고리 크롤링 중:  (CATE_CD=104569)

🔍 카테고리 크롤링 중:  (CATE_CD=104539)

🔍 카테고리 크롤링 중:  (CATE_CD=104542)

🔍 카테고리 크롤링 중:  (CATE_CD=104545)

🔍 카테고리 크롤링 중:  (CATE_CD=104548)

🔍 카테고리 크롤링 중:  (CATE_CD=104551)

🔍 카테고리 크롤링 중:  (CATE_CD=104554)

🔍 카테고리 크롤링 중:  (CATE_CD=104557)

🔍 카테고리 크롤링 중:  (CATE_CD=104560)

🔍 카테고리 크롤링 중:  (CATE_CD=104563)

🔍 카테고리 크롤링 중:  (CATE_CD=104566)

🔍 카테고리 크롤링 중:  (CATE_CD=104573)

🔍 카테고리 크롤링 중:  (CATE_CD=104577)

🔍 카테고리 크롤링 중:  (CATE_CD=104580)

🔍 카테고리 크롤링 중:  (CATE_CD=104586)

✅ 저장 완료: kpi_전기자재_카테고리별_상품.csv


In [28]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd, os

# ──────────────────────────────────────────────────────────────
# 1. KPI 로그인 계정
USER_ID = os.getenv("KPI_ID", "leewu08")            # ← 계정 입력
USER_PW = os.getenv("KPI_PW", "Emporio119")      # ← 비번 입력
START_URL = "https://www.kpi.or.kr/www/price/category.asp?CATE_CD=104501"

# 2. Selenium WebDriver 설정
opt = Options()
opt.add_argument("--start-maximized")
# opt.add_argument("--headless")                    # GUI 없이 돌리고 싶으면 주석 해제
driver = webdriver.Chrome(options=opt)
wait   = WebDriverWait(driver, 12)

try:
    # ──────────────────────────────────────────────────────────
    # 3. 로그인
    driver.get(START_URL)
    wait.until(EC.element_to_be_clickable(
        (By.XPATH, '//a[contains(text(),"로그인")]'))
    ).click()
    wait.until(EC.visibility_of_element_located(
        (By.CSS_SELECTOR, "#user_id"))
    ).send_keys(USER_ID)
    driver.find_element(By.CSS_SELECTOR, "#user_pw").send_keys(USER_PW)
    driver.find_element(By.CSS_SELECTOR, "#sendLogin").click()

    # 사이드 메뉴가 표시될 때까지 대기
    wait.until(EC.presence_of_element_located((By.ID, "location")))
    print("✅  로그인 성공")

    # ──────────────────────────────────────────────────────────
    # 4. 사이드바 <a> 태그 전부 → 딕셔너리 리스트화
    cat_dicts = []
    for a in driver.find_elements(
            By.CSS_SELECTOR, "#location > div.dep-two > dl > dd > ul li a"):

        name = a.text.strip()
        raw  = a.get_attribute("href") or ""
        if not name:
            continue                               # 빈 <li> 건너뜀

        # href 두 가지 유형 처리
        if   "CATE_CD="   in raw:
            cate_cd = raw.split("CATE_CD=")[-1].split("'")[0]
        elif "openCate('" in raw:                  # javascript:openCate('104560')
            cate_cd = raw.split("openCate('")[-1].split("'")[0]
        else:
            continue

        url = f"https://www.kpi.or.kr/www/price/category.asp?CATE_CD={cate_cd}"
        cat_dicts.append({"cate_cd": cate_cd, "name": name, "url": url})

    print(f"📦  카테고리 {len(cat_dicts)}개 수집")

    # ──────────────────────────────────────────────────────────
    # 5. 카테고리별 상품 수집
    rows = []
    for cat in cat_dicts:
        print(f"\n🔍  {cat['name']} ({cat['cate_cd']}) 진입")
        driver.get(cat["url"])

        # 상품 블록 존재 여부 확인
        try:
            wait.until(EC.presence_of_element_located(
                (By.CSS_SELECTOR, "#table_two_sub > div.goods-item div.set-mitem")
            ))
        except:
            print("    ↳ 상품 없음, skip");  continue

        for itm in driver.find_elements(
                By.CSS_SELECTOR, "#table_two_sub > div.goods-item div.set-mitem"):

            try:
                a_tag = itm.find_element(By.TAG_NAME, "a")
                code  = a_tag.get_attribute("href").split("openGoods('")[1].split("')")[0]
                title = a_tag.get_attribute("title")
                img   = itm.find_element(By.CSS_SELECTOR, "li.set-img img").get_attribute("src")
                if img.startswith("/"):                 # 상대경로 → 절대경로
                    img = "https://www.kpi.or.kr" + img

                rows.append({
                    "카테고리": cat["name"],            # ← 처음 저장해 둔 카테고리명 그대로
                    "제품코드": code,
                    "제품명"  : title,
                    "이미지"  : img
                })
            except Exception as e:
                print(f"    [!] 파싱 오류: {e}")

    # ──────────────────────────────────────────────────────────
    # 6. CSV 저장
    if rows:
        pd.DataFrame(rows).to_csv(
            "kpi_전기자재_카테고리별_상품.csv",
            index=False, encoding="utf-8-sig"
        )
        print("\n✅  CSV 저장 완료 → kpi_전기자재_카테고리별_상품.csv")
    else:
        print("\n⚠️  수집된 데이터가 없습니다.")

finally:
    driver.quit()


ElementClickInterceptedException: Message: element click intercepted: Element <input type="image" id="sendLogin" src="../images/member/login_btn_img.gif" width="480" border="0" tabindex="8"> is not clickable at point (692, 676). Other element would receive the click: <img src="/www/2020/images/banner/kosme_topban.png" alt="">
  (Session info: chrome=137.0.7151.120); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#elementclickinterceptedexception
Stacktrace:
	GetHandleVerifier [0x0x7ff75737cda5+78885]
	GetHandleVerifier [0x0x7ff75737ce00+78976]
	(No symbol) [0x0x7ff757139bca]
	(No symbol) [0x0x7ff757198779]
	(No symbol) [0x0x7ff757196112]
	(No symbol) [0x0x7ff757193151]
	(No symbol) [0x0x7ff757192041]
	(No symbol) [0x0x7ff757183654]
	(No symbol) [0x0x7ff7571b8b8a]
	(No symbol) [0x0x7ff757182f06]
	(No symbol) [0x0x7ff7571b8da0]
	(No symbol) [0x0x7ff7571e122f]
	(No symbol) [0x0x7ff7571b8963]
	(No symbol) [0x0x7ff757181

In [17]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
import time
import pandas as pd

# --- 사용자 계정 입력 ---
USER_ID = "leewu08"         # ← 여기에 본인 KPI ID 입력
USER_PW = "Emporio119"   # ← 여기에 본인 KPI 비밀번호 입력

# --- Chrome 설정 ---
options = Options()
options.add_argument("--start-maximized")
# options.add_argument("--headless")  # UI 없이 돌리고 싶으면 주석 해제

driver = webdriver.Chrome(options=options)

try:
    # 1. 로그인 페이지로 진입
    driver.get("https://www.kpi.or.kr/www/price/category.asp?CATE_CD=104501")
    time.sleep(2)

    # 2. 로그인
    driver.find_element(By.XPATH, '//a[contains(text(), "로그인")]').click()
    time.sleep(2)
    driver.find_element(By.CSS_SELECTOR, "#user_id").send_keys(USER_ID)
    driver.find_element(By.CSS_SELECTOR, "#user_pw").send_keys(USER_PW)
    driver.find_element(By.CSS_SELECTOR, "#sendLogin").click()
    time.sleep(3)

    # 3. 다시 메인 카테고리 진입 (DOM 재렌더링 이후)
    driver.get("https://www.kpi.or.kr/www/price/category.asp?CATE_CD=104501")
    time.sleep(2)

    # 4. 카테고리 링크 수집
    category_links = driver.find_elements(By.CSS_SELECTOR, "#location > div.dep-two > dl > dd > ul li a")
    category_list = []

    for link in category_links:
        name = link.text.strip()
        href = link.get_attribute("href")  # 상대경로: /www/price/category.asp?CATE_CD=104501
        if href and "CATE_CD=" in href:
            cate_cd = href.split("CATE_CD=")[-1]
            category_list.append({
                "name": name if name else "(미확인)",
                "cate_cd": cate_cd
            })

    print(f"\n📦 수집된 카테고리 개수: {len(category_list)}")

    # 5. 카테고리별 순회하며 상품 수집
    results = []

    for cat in category_list:
        print(f"\n🔍 카테고리: {cat['name']} (CATE_CD={cat['cate_cd']})")
        url = f"https://www.kpi.or.kr/www/price/category.asp?CATE_CD={cat['cate_cd']}"
        driver.get(url)
        time.sleep(2)

        items = driver.find_elements(By.CSS_SELECTOR, "#table_two_sub > div.goods-item div.set-mitem")
        print(f"  - 상품 수: {len(items)}")

        for item in items:
            try:
                a_tag = item.find_element(By.TAG_NAME, "a")
                href = a_tag.get_attribute("href")
                code = href.split("openGoods('")[1].split("')")[0]
                title = a_tag.get_attribute("title")

                img_tag = item.find_element(By.CSS_SELECTOR, "li.set-img img")
                img_url = img_tag.get_attribute("src")
                if img_url.startswith("/"):
                    img_url = "https://www.kpi.or.kr" + img_url

                results.append({
                    "카테고리": cat["name"],
                    "제품코드": code,
                    "제품명": title,
                    "이미지": img_url
                })

            except Exception as e:
                print(f"[!] 오류 - {cat['name']} 상품: {e}")
                continue

    # 6. 결과 저장
    df = pd.DataFrame(results)
    df.to_csv("kpi_전기자재_카테고리별_상품.csv", index=False, encoding="utf-8-sig")
    print("\n✅ 저장 완료: kpi_전기자재_카테고리별_상품.csv")

finally:
    driver.quit()


📦 수집된 카테고리 개수: 26

🔍 카테고리: (미확인) (CATE_CD=104501)
  - 상품 수: 10

🔍 카테고리: (미확인) (CATE_CD=104504)
  - 상품 수: 7

🔍 카테고리: (미확인) (CATE_CD=104507)
  - 상품 수: 2

🔍 카테고리: (미확인) (CATE_CD=104513)
  - 상품 수: 13

🔍 카테고리: (미확인) (CATE_CD=104516)
  - 상품 수: 4

🔍 카테고리: (미확인) (CATE_CD=104587)
  - 상품 수: 0

🔍 카테고리: (미확인) (CATE_CD=104521)
  - 상품 수: 3

🔍 카테고리: (미확인) (CATE_CD=104524)
  - 상품 수: 6

🔍 카테고리: (미확인) (CATE_CD=104526)
  - 상품 수: 3

🔍 카테고리: (미확인) (CATE_CD=104530)
  - 상품 수: 12

🔍 카테고리: (미확인) (CATE_CD=104533)
  - 상품 수: 5

🔍 카테고리: (미확인) (CATE_CD=104569)
  - 상품 수: 2

🔍 카테고리: (미확인) (CATE_CD=104539)
  - 상품 수: 4

🔍 카테고리: (미확인) (CATE_CD=104542)
  - 상품 수: 4

🔍 카테고리: (미확인) (CATE_CD=104545)
  - 상품 수: 0

🔍 카테고리: (미확인) (CATE_CD=104548)
  - 상품 수: 1

🔍 카테고리: (미확인) (CATE_CD=104551)
  - 상품 수: 4

🔍 카테고리: (미확인) (CATE_CD=104554)
  - 상품 수: 6

🔍 카테고리: (미확인) (CATE_CD=104557)
  - 상품 수: 6

🔍 카테고리: (미확인) (CATE_CD=104560)
  - 상품 수: 9

🔍 카테고리: (미확인) (CATE_CD=104563)
  - 상품 수: 5

🔍 카테고리: (미확인) (CATE_CD=104566)
  - 상품 수: 7

🔍 카테고리: (

KeyboardInterrupt: 

In [32]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd, os

# ── 계정 ─────────────────────────────────────────
USER_ID = os.getenv("KPI_ID", "leewu08")
USER_PW = os.getenv("KPI_PW", "Emporio119")

# ── 카테고리 고정 목록 ───────────────────────────
categories = [
    {"cate_cd": "104501", "name": "절연전선"},
    {"cate_cd": "104504", "name": "전력케이블"},
    {"cate_cd": "104507", "name": "제어용케이블"},
    {"cate_cd": "104513", "name": "소방용케이블"},
    {"cate_cd": "104516", "name": "통신용케이블"},
    {"cate_cd": "104587", "name": "기타특수케이블"},
    {"cate_cd": "104521", "name": "전선원부자재"},
    {"cate_cd": "104524", "name": "전선관"},
    {"cate_cd": "104526", "name": "전선관로재"},
    {"cate_cd": "104530", "name": "전력기기"},
    {"cate_cd": "104533", "name": "배전기기"},
    {"cate_cd": "104569", "name": "절연재료"},
    {"cate_cd": "104539", "name": "수ㆍ배전반"},
    {"cate_cd": "104542", "name": "배전제어기기"},
    {"cate_cd": "104545", "name": "자동화기기"},
    {"cate_cd": "104548", "name": "계전기기"},
    {"cate_cd": "104551", "name": "접지ㆍ피뢰자재"},
    {"cate_cd": "104554", "name": "배선기구"},
    {"cate_cd": "104557", "name": "조명기구"},
    {"cate_cd": "104560", "name": "전등"},
    {"cate_cd": "104563", "name": "안정기"},
    {"cate_cd": "104566", "name": "전주"},
    {"cate_cd": "104573", "name": "가선철물"},
    {"cate_cd": "104577", "name": "교통신호장치"},
    {"cate_cd": "104580", "name": "전광게시물"},
    {"cate_cd": "104586", "name": "전지"},
]

# ── 드라이버 ─────────────────────────────────────
opt = Options(); opt.add_argument("--start-maximized")
driver = webdriver.Chrome(options=opt)
wait   = WebDriverWait(driver, 12)

try:
    # 로그인
    driver.get("https://www.kpi.or.kr/www/price/category.asp?CATE_CD=104501")
    wait.until(EC.element_to_be_clickable(
        (By.XPATH,'//a[contains(text(),"로그인")]'))).click()
    wait.until(EC.visibility_of_element_located(
        (By.CSS_SELECTOR,"#user_id"))).send_keys(USER_ID)
    driver.find_element(By.CSS_SELECTOR,"#user_pw").send_keys(USER_PW)
    driver.find_element(By.CSS_SELECTOR,"#sendLogin").click()
    rows = []

    # 고정 categories 순회
    for cat in categories:
        url = f"https://www.kpi.or.kr/www/price/category.asp?CATE_CD={cat['cate_cd']}"
        print(f"\n🔍 {cat['name']} ({cat['cate_cd']})")
        driver.get(url)

        # 상품 블록 확인
        try:
            wait.until(EC.presence_of_element_located(
                (By.CSS_SELECTOR,"#table_two_sub > div.goods-item div.set-mitem")))
        except:
            print("   ↳ 상품 없음"); continue

        for itm in driver.find_elements(
                By.CSS_SELECTOR,"#table_two_sub > div.goods-item div.set-mitem"):
            try:
                a_tag = itm.find_element(By.TAG_NAME,"a")
                code  = a_tag.get_attribute("href").split("openGoods('")[1].split("')")[0]
                title = a_tag.get_attribute("title")
                img   = itm.find_element(By.CSS_SELECTOR,"li.set-img img").get_attribute("src")
                if img.startswith("/"): img = "https://www.kpi.or.kr"+img
                rows.append({
                    "카테고리": cat["name"],
                    "제품코드": code,
                    "제품명"  : title,
                    "이미지"  : img
                })
            except Exception as e:
                print(f"   [!] 파싱 오류: {e}")

    # CSV 저장
    if rows:
        pd.DataFrame(rows).to_csv("kpi_전기자재_카테고리별_상품.csv",
                                  index=False, encoding="utf-8-sig")
        print("\n✅ CSV 저장 완료!")
    else:
        print("\n⚠️ 데이터 없음")

finally:
    driver.quit()



🔍 절연전선 (104501)

🔍 전력케이블 (104504)

🔍 제어용케이블 (104507)

🔍 소방용케이블 (104513)

🔍 통신용케이블 (104516)

🔍 기타특수케이블 (104587)
   ↳ 상품 없음

🔍 전선원부자재 (104521)

🔍 전선관 (104524)

🔍 전선관로재 (104526)

🔍 전력기기 (104530)

🔍 배전기기 (104533)

🔍 절연재료 (104569)

🔍 수ㆍ배전반 (104539)

🔍 배전제어기기 (104542)

🔍 자동화기기 (104545)
   ↳ 상품 없음

🔍 계전기기 (104548)

🔍 접지ㆍ피뢰자재 (104551)

🔍 배선기구 (104554)

🔍 조명기구 (104557)

🔍 전등 (104560)

🔍 안정기 (104563)

🔍 전주 (104566)

🔍 가선철물 (104573)

🔍 교통신호장치 (104577)

🔍 전광게시물 (104580)

🔍 전지 (104586)

✅ CSV 저장 완료!
